# 第七课｜第一个 RTL 神经元

今天把 state/clock、Boolean logic 和 RTL 拼起来：
> **把一次 neuron update 拆成 combinational path 与 sequential update。**

为避免偷选尚未冻结的 LIF/fixed-point 细节，本课使用**教学用 integrate-and-fire neuron**，直接复用第四课 accumulator + threshold + reset；它不是正式 `MOD-003`。


## 1. 本课 contract

每个 cycle：读旧 `membrane_v` → `candidate = membrane_v + input_current` → `candidate >= threshold` 产生 spike → spike 时 next state 为 reset，否则为 candidate → clock edge 才保存。

测试向量避开 overflow。正式 leak/rounding/overflow 仍由 RMD-002/RMD-003 冻结。


In [ ]:
def tutorial_if_step(state, input_current, threshold, reset_value=0):
    candidate = state + input_current
    spike = candidate >= threshold
    return (reset_value if spike else candidate), spike, candidate

state = 0
for cycle, current in enumerate([1,1,1,1,2,2]):
    nxt, spike, candidate = tutorial_if_step(state,current,4)
    print(f'cycle={cycle}: state={state}, input={current}, candidate={candidate}, spike={spike}, next={nxt}')
    state = nxt


## 2. 结构图

```mermaid
flowchart LR
 REG["membrane_v register"] --> ADD["adder"]
 IN["input_current"] --> ADD
 ADD --> C["candidate"]
 C --> CMP[">= threshold"]
 TH["threshold"] --> CMP
 CMP --> MUX["reset or candidate"]
 C --> MUX
 R["reset_value"] --> MUX
 MUX --> REG
 CLK["clock edge"] -.-> REG
```


## 3. `always_comb` 与 `always_ff`

`always_comb` 描述 next-state 的组合计算；`always_ff` 描述 clock edge 时保存 state。`candidate_ext` / `spike_next` / `next_v` 是本 cycle 的计算，`membrane_v` 是跨 cycle 保存的 state。


## 4. 完整教学 RTL

```systemverilog
module tutorial_if_neuron #(
    parameter int WIDTH = 8
) (
    input  logic clk,
    input  logic rst_n,
    input  logic signed [WIDTH-1:0] input_current,
    input  logic signed [WIDTH-1:0] threshold,
    input  logic signed [WIDTH-1:0] reset_value,
    output logic signed [WIDTH-1:0] membrane_v,
    output logic spike
);
    logic signed [WIDTH:0] candidate_ext;
    logic signed [WIDTH-1:0] next_v;
    logic spike_next;

    always_comb begin
        candidate_ext = $signed({membrane_v[WIDTH-1], membrane_v})
                      + $signed({input_current[WIDTH-1], input_current});
        spike_next = candidate_ext >= $signed({threshold[WIDTH-1], threshold});
        next_v = spike_next ? reset_value : candidate_ext[WIDTH-1:0];
    end

    always_ff @(posedge clk) begin
        if (!rst_n) begin
            membrane_v <= reset_value;
            spike <= 1'b0;
        end else begin
            membrane_v <= next_v;
            spike <= spike_next;
        end
    end
endmodule
```


## 5. 为什么 candidate 多一 bit？

两个 WIDTH-bit signed 数相加可能需要多一 bit，所以先放在 `candidate_ext[WIDTH:0]`。本教学模块若 non-spiking next state 超出 WIDTH 会截断；当前测试不触发这种情况。正式 RTL 必须服从未来冻结的 overflow policy。


## 6. Try It

edge 前 `membrane_v=3, input=1, threshold=4, reset=0`。先写出 candidate、spike_next、next_v，再写 edge 后 membrane_v/spike。


## 7. AI Task / Human Check

让 AI 对照五条 contract 逐条指出 RTL 对应位置，不允许它改 contract。

不用 AI 应能指出 combinational path、真正的 state，以及解释为什么 `candidate=4` 与 edge 后 `membrane_v=0` 可以同时正确。


## 8. Engineering Handoff / Project Trace

`rtl/learning/tutorial_if_neuron.sv` 是 RMD-004 前的教学实现；正式 `rtl/neuron/lif_neuron_engine.sv` 仍等待 v0 semantics 与 fixed-point contract。

- Lesson: `LSN-007`
- Mapping: `RMD-004` teaching precursor
- Formal `MOD-003`: intentionally not created


## 9. Exit Ticket

你能从 contract 画出 combinational path + register，并从 RTL 指出两部分。下一课不再增加 neuron 行为，只验证它。
